# Study 932 — Trust Yield 🏦

**A pre-deal SPAC quoted below its trust: a T-bill on sale, or a trap?**

A SPAC is a pot of Treasuries with a deadline. It raises about **$10 a share** into a
trust, and every public share carries a *redemption right* — hand the share back at the
deal vote (or at liquidation) and collect your pro-rata slice of that trust, plus the
interest it earned. The right does not depend on the deal failing, and it does not depend
on how you vote.

So a pre-deal SPAC quoted **below** trust is a Treasury bought at a discount. Annualise
that discount over the time left, and you get a yield *on top of* the bill yield — with
any deal upside thrown in for free.

We test the flat version of that trade on **31 hardcoded 2019-2022-vintage
SPACs** (2019-03-27 → 2024-04-10) against **BIL**: each month-end, buy every
shell quoted under trust, execute at the **next** close, hold to the redemption deadline,
**redeem at trust**. 15 bps one-way, no shorting, one execution lag.

*Every real number below is the frozen headline from `docs/results.md` (fingerprint
`7d2c2f7b9919`, as-of 2026-06-30). Live cells are labelled as such.*


## 1. What you are actually buying

Forget the target company, the SPAC mania, the celebrity sponsor. Before a deal closes, the thing you own is a claim on a pile of short-dated US Treasuries and a contractual right to ask for it back on a known date. If the market sells you that claim for less than the pile is worth, the arithmetic does the rest.

Here is what the tape actually offered, on average, when it was below trust:

In [1]:
R = {'disc': 1.47, 'hold_days': 302, 'ytr': 4.03, 'exc': 1.34, 'book': 1.23, 'hit': 93.1, 'n_pos': 203}
print(f"average discount to trust at entry : {R['disc']:+.2f}%")
print(f"average time left to redemption    : {R['hold_days']:.0f} days")
print(f"=> implied annualised yield-to-redemption: {R['ytr']:+.2f}%")
print(f"what it actually paid, over cash   : {R['exc']:+.2f}% per position")
print(f"as a book, per year over T-bills   : {R['book']:+.2f}%")
print(f"positions that made money          : {R['hit']:.1f}% of {R['n_pos']}")

average discount to trust at entry : +1.47%
average time left to redemption    : 302 days
=> implied annualised yield-to-redemption: +4.03%
what it actually paid, over cash   : +1.34% per position
as a book, per year over T-bills   : +1.23%
positions that made money          : 93.1% of 203


## 2. The answer: yes, and it is small

Of the 31 shells on the list, **25** ever traded below trust. Every single one of those 25 paid — a **100%** hit rate on the one-bet-per-name cut. But look at the size of the prize: about **1.2% a year over T-bills**, for money locked up for the best part of a year in a shell company nobody was trading.

> ⚠️ **The honest caveat, up front.** This study *pays itself the trust*: the model assumes a redeemed share hands back its accrued $10, because that is what the contract says. So the headline is closer to arithmetic than to a discovery — what the tape genuinely establishes is that the discount to that line was real and that **the line itself was where the market was** (on the day the redemption right expired, 14 of the 25 shells were quoted *at or above* it, and the worst was only 2.9% under). Notebook 02 does the arithmetic in public.

> 🔬 **For the quants.** The honest interval is a cluster bootstrap over the 25 names, not a *t* on the 203 overlapping monthly entries: 95% CI **[+1.04%, +1.63%]** per position, zero negative resamples. The synthetic null in notebook 02 shows exactly why the naive *t* of +17.9 must not be quoted.

## 3. Why the discount was there at all

In 2021 there was no discount — there was a **premium**. Money piled into pre-deal shells hoping for the next hot merger; in 2021-02 the median shell on this list traded **9.4% above** its trust. Buying a Treasury for eleven dollars is not an arbitrage.

Then the mood broke and rates rose. Holders wanted out, there were no buyers, and by **2022-08** the median live shell was **+2.0% below** trust — an implied **7.1%** yield-to-redemption against a **2.9%** three-month bill. That is the whole trade, and it lasted about eighteen months.

## 4. The four things that could have gone wrong

1. **The trust might not be $10.** It is the assumption the whole result rests on. At $9.90 the edge more than halves (+0.56% a year); at $10.20 it triples. We do not know each shell's filed figure — we assume $10.00 and sweep it. The sharpest version of this test is to let the market overrule us shell by shell: pay the *worse* of the assumed trust and what the shell was actually quoted at on the deadline. The edge survives that, at +0.83% a position (+0.86% a year).
2. **The deadline might move.** 2022-2023 was the era of serial extension votes. Each extension is *also* a redemption chance, so it shortens the realised wait — but it turns a dated bill into an open-ended one.
3. **The spread might eat it.** At 100 bps one-way the edge is down to +0.30% a year; at 200 bps it is **negative**. One to three cents on a $9.80 quote is not a fantasy in size.
4. **There might be nothing to buy.** Live pre-deal shells on this list: 16 in early 2021, 5 by end-2022, 1 by 2024. The market died.

## 5. Live check — the machinery is not inventing this (offline synthetic)

Buying below a line and being paid that line is arithmetic, so the only null worth running is one where the **line is a fiction**: quotes wander with no anchor to any trust, and what you are paid at the end is simply the last quote. In that world the rule must earn nothing. Six worlds of each kind, run live below.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
from trust_yield import data, strategy as st
for ss, label in [(1.0, 'trust put binds  '), (0.0, 'put is a fiction ')]:
    outs = [st.synthetic_detect(*data.synthetic_panel(signal_strength=ss, seed=s,
                                                     n_days=700)[:3], n_boot=600)
            for s in range(932, 938)]
    m = np.array([o['mean_excess'] for o in outs])
    fires = sum(1 for o in outs if o['ci_low'] > 0)
    print(f"{label}: mean excess {m.mean():+.2%}  "
          f"(worlds where the edge is significant: {fires}/6)")
print('\n(synthetic, not the tape — any single null world can fire by luck; '
      'notebook 02 runs the rate over 16)')

trust put binds  : mean excess +1.86%  (worlds where the edge is significant: 6/6)


put is a fiction : mean excess -0.01%  (worlds where the edge is significant: 0/6)

(synthetic, not the tape — any single null world can fire by luck; notebook 02 runs the rate over 16)


## Verdict

- **Signal — Real.** The discount existed: **+1.34%** per 302-day position, cluster-bootstrap CI **[+1.04%, +1.63%]** across 25 shells with no negative resamples, positive in both rate eras. That number is an **identity** — we pay ourselves the trust by assumption — so the stamp leans on the parts that are not assumed: the deadline-day quote sat at or above the assumed trust line in 14/25 shells and never more than 2.9% under it, and paying the *worse* of the two still leaves +0.83% a position. **Survivorship:** this is a list of shells whose successor ticker still trades and which never did a post-deal reverse split, so the *count of chances* is flattered — though the payoff itself (the trust) is what a liquidation pays too.
- **Tradability — Fragile.** +1.2% a year over bills as a book, for a 302-day lock-up in a shell trading a few hundred thousand dollars a day. It dies at 200 bps of friction, it more than halves if the trust was $9.90, it assumes a deadline that sponsors kept moving, the broker's fee for filing the redemption is not modelled at all, and the opportunity set collapsed from 16 live shells to 1. A real mechanism you could not have sized, in a market that no longer exists.